<a href="https://colab.research.google.com/github/anastasijaana/MD/blob/main/Lab_1_5_Incident%C5%B3_klasifikavimo_Agentas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lab 1.5: Incidentų klasifikavimo AI Agentas (Incidentų Klasifikavimo Tarnyba)
Autorė: Anastasija Golubeva, ISIfm-24

Darbo tema: Ontologija ir duomenimis pagrįsto verslo procesų modeliavimo tyrimas

Sistemos Aprašymas
Šis sąsiuvinis demonstruoja "end-to-end" AI sprendimą, paremtą magistro darbu. Sistema veikia kaip „Incidentų Klasifikavimo Tarnyba“ , skirta IT paslaugų tiekėjui.

Agento tikslas – gauti laisvu tekstu pateiktą incidento aprašymą (X) ir automatiškai priskirti jam struktūrizuotą, ontologija paremtą klasifikaciją – Kategoriją ir Subkategoriją (Y).

Šioje demonstracijoje naudojama Google Gemini API (kaip GAI modelis) ir „Prompt Engineering“ metodai (parengti 1.4 laboratoriniame darbe), kad būtų įgyvendintas pilnas sprendimo ciklas.

Kodas atlieka saugų prisijungimą prie Google Gemini dirbtinio intelekto paslaugos Google Colab aplinkoje (įdiegiama Google AI biblioteka, importuojamos bibliotekos ir užkraunamas API raktas)

In [ ]:
!pip install -q google-generativeai

import google.generativeai as genai
import os
from google.colab import userdata
import json

# Saugus API rakto gavimas iš Colab Secrets
try:
    GEMINI_KEY = userdata.get('GEMINI_KEY')
    if GEMINI_KEY is None:
        raise ValueError("GEMINI_KEY nerastas. Ar nustatėte jį 'Secrets' skiltyje?")

    genai.configure(api_key=GEMINI_KEY)
    print("✅ Sėkmingai konfigūruota su Gemini API.")

except Exception as e:
    print(f"❌ KLAIDA: {e}")
    print("Prašome patikrinti API rakto nustatymus 'Tools -> Secrets'.")

✅ Sėkmingai konfigūruota su Gemini API.


Konfigūruojamas modelis

In [15]:
# Konfigūracija, nurodanti modeliui grąžinti TIK JSON
json_output_config = genai.GenerationConfig(
    response_mime_type="application/json"
)

# Inicijuojame modelį su šia konfigūracija
# Naudojame 'flash' modelį greičiui
model = genai.GenerativeModel(
    model_name="gemini-2.5-pro",
    generation_config=json_output_config
)

print(f"Paruoštas modelis: {model.model_name}")

Paruoštas modelis: models/gemini-2.5-pro


Demonstracija 1: Zero-Shot Užklausa
Zero-Shot: Modeliui pateikiama užduotis be jokių išankstinių pavyzdžių.

Tikslas: Patikrinti, kaip modelis susidoroja su nauja užduotimi, remdamasis tik rolės apibrėžimu.

In [ ]:
# Apibrėžiame rolę ir užduotį.
# Prašome JSON formato, kurį jau esame nustatę ir per 'generation_config'.
prompt_zero_shot = """
Tu esi dirbtinio intelekto agentas, veikiantis kaip „Incidentų klasifikavimo tarnyba“ dideliam IT paslaugų tiekėjui. Tavo užduotis – gauti laisvu tekstu parašytą incidento aprašymą (X) ir, remiantis incidento valdymo ontologija, priskirti jam teisingą `Kategorija` ir `Subkategorija` (Y).

Tavo atsakymas PRIVALO būti pateiktas JSON formatu:
{
  "kategorija": "...",
  "subkategorija": "..."
}

---
**Incidento aprašymas (X):**
"Nepavyksta prisijungti prie VPN tinklo. Vakar viskas veikė, šiandien meta klaidą."
"""

# Incidento įvestis (X)
input_X_1 = "Nepavyksta prisijungti prie VPN tinklo. Vakar viskas veikė, šiandien meta klaidą."

Prompt 1 Vykdymas ir Rezultatas (Y)

In [16]:
print("Siunčiama 'Zero-Shot' užklausa modeliui...")

try:
    # Siunčiame promptą modeliui
    response = model.generate_content(prompt_zero_shot)

    # Išgauname Y
    output_Y_1_raw = response.text

    # --- Vizualizacija (Užduotis 1.5.7) ---
    print("\n--- 🟢 REZULTATAS (Zero-Shot) ---")
    print(f"\n➡️ ĮVESTIS (X):\n{input_X_1}")

    # Kadangi paprašėme JSON, bandome jį atspausdinti gražiai
    try:
        output_Y_1_json = json.loads(output_Y_1_raw)
        print(f"\n⬅️ IŠVESTIS (Y) (JSON formatu):\n{json.dumps(output_Y_1_json, indent=2, ensure_ascii=False)}")
    except json.JSONDecodeError:
        print(f"\n⬅️ IŠVESTIS (Y) (Gautas ne-JSON tekstas):\n{output_Y_1_raw}")

except Exception as e:
    print(f"\n❌ KLAIDA vykdant užklausą: {e}")

Siunčiama 'Zero-Shot' užklausa modeliui...

--- 🟢 REZULTATAS (Zero-Shot) ---

➡️ ĮVESTIS (X):
Nepavyksta prisijungti prie VPN tinklo. Vakar viskas veikė, šiandien meta klaidą.

⬅️ IŠVESTIS (Y) (JSON formatu):
{
  "kategorija": "Tinklas",
  "subkategorija": "VPN"
}


Demonstracija 2: Few-Shot Užklausa

Few-Shot: Modeliui pateikiami keli (X, Y) pavyzdžiai, kad jis geriau suprastų užduotį ir formatą.

Tikslas: Padidinti modelio tikslumą, suteikiant jam kontekstinių pavyzdžių, kurie veikia kaip "mini-mokymas". Tai ypač svarbu, kai norime atskirti panašias kategorijas (pvz., "Wi-Fi" vs "VPN", kurios abi yra "Tinklo problemos", bet skirtingos subkategorijos ).

In [17]:
prompt_few_shot = """
Tu esi dirbtinio intelekto agentas, veikiantis kaip „GAI klasifikavimo tarnyba“. Tavo užduotis – gauti laisvu tekstu parašytą incidento aprašymą (X) ir, remiantis incidento valdymo ontologija, priskirti jam teisingą `Kategorija` ir `Subkategorija` (Y).

Tavo atsakymas PRIVALO būti pateiktas JSON formatu.

Štai keli pavyzdžiai, kaip tu atlieki šią užduotį:

**Pavyzdys 1:**
* **X (Aprašymas):** "Neveikia spausdintuvas kabinete 302." [cite: 193]
* **Y (Rezultatas):**
    {
      "kategorija": "Spausdinimo įranga",
      "subkategorija": "Spausdintuvas"
    } [cite: 196]

**Pavyzdys 2:**
* **X (Aprašymas):** "Dingo Wi-Fi ryšys visame pastate, negalime dirbti."
* **Y (Rezultatas):**
    {
      "kategorija": "TinkloProblemos",
      "subkategorija": "WiFiProblem"
    } [cite: 45, 46]

**Pavyzdys 3:**
* **X (Aprašymas):** "Nepavyksta prisijungti prie VPN"
* **Y (Rezultatas):**
    {
      "kategorija": "TinkloIštekliai",
      "subkategorija": "VPN"
    } [cite: 83]


**DABARTINĖ UŽDUOTIS:**

Dabar, remdamasis šiais pavyzdžiais, atlik klasifikavimą naujam incidentui.

**Incidento aprašymas (X):**
"Mano kompiuteris labai lėtai veikia, o atidarant el. paštą iššoka keista lentelė apie užrakintus failus. Įtariu virusą."
"""

# Incidento įvestis (X)
input_X_2 = "Mano kompiuteris labai lėtai veikia, o atidarant el. paštą iššoka keista lentelė apie užrakintus failus. Įtariu virusą."

Prompt 2 Vykdymas ir Rezultatas (Y)

In [18]:
print("Siunčiama 'Few-Shot' užklausa modeliui...")

try:
    # Siunčiame promptą modeliui
    response = model.generate_content(prompt_few_shot)

    # Išgauname Y
    output_Y_2_raw = response.text

    # --- Vizualizacija (Užduotis 1.5.7) ---
    print("\n--- 🟢 REZULTATAS (Few-Shot) ---")
    print(f"\n➡️ ĮVESTIS (X):\n{input_X_2}")

    # Kadangi paprašėme JSON, bandome jį atspausdinti gražiai
    try:
        output_Y_2_json = json.loads(output_Y_2_raw)
        print(f"\n⬅️ IŠVESTIS (Y) (JSON formatu):\n{json.dumps(output_Y_2_json, indent=2, ensure_ascii=False)}")
    except json.JSONDecodeError:
        print(f"\n⬅️ IŠVESTIS (Y) (Gautas ne-JSON tekstas):\n{output_Y_2_raw}")

except Exception as e:
    print(f"\n❌ KLAIDA vykdant užklausą: {e}")

Siunčiama 'Few-Shot' užklausa modeliui...

--- 🟢 REZULTATAS (Few-Shot) ---

➡️ ĮVESTIS (X):
Mano kompiuteris labai lėtai veikia, o atidarant el. paštą iššoka keista lentelė apie užrakintus failus. Įtariu virusą.

⬅️ IŠVESTIS (Y) (JSON formatu):
{
  "kategorija": "SaugumoIncidentas",
  "subkategorija": "KenkėjiškaPrograma"
}
